# PDB to ODE and NERDSS Workflow
## Tutorial 3 : benzaldehyde lyase mutant M6 from Herbiconiux sp. SALV-R1 (8y7s)

This tutorial demonstrates 8y7s as an example to use ProAffinity-GNN(1) to predict binding affinities. Note that for detailed explanation of the workflow, refer to tutorial 1: `ionerdss_tutorial_6bno.ipynb`

> ### Aims for this tutorial:
> - Give an example of a homo-tetramer assembly but has three different types of symmetric bindings
> - Tutorial on how to install and use ProAffinity-GNN to predict binding affinities

## Example: 8Y7S Structure

- In this file, we'll use **8Y7S** - HIV-1 CA-SP1 assembly as our example.

---
## Part 1: Preparation - Install Dependencies

ProAffinity-GNN(1) depends on PyTorch and AutoDockFR (ADFR)(2,3). The installation is specific to the hardware, conda/pip/wheel, the CUDA version, and the operating system. Therefore, we keep PyTorch and ADFR out of the default dependency files and the users need to install them separately.

### 1.1. Install optional dependencies (PyTorch (v2.2.2))

We suggest the user create a separate environment for ioNERDSS with proaffinity-gnn due to strict dependency version requirement. In the new envrionment, install ioNERDSS with optional dependencies for proaffinity-gnn via:

```
pip install "ioNERDSS[proaffinity]"
```

You can still use the same command if you do not want to create a new environment but be aware of the potential version conflict.

> **Note:** The ProAffinity-GNN is written and tested with pyTorch v2.2.2, though later versions may also work.

> **WARNING:** PyTorch v2.2.2 requires `numpy` version < 2.0.0. The lower version is installed by default with the provided conda environment. If you have a higher version of `numpy`, it is suggest to create a new conda environment with `numpy` version < 2.0.0 and install PyTorch again to avoid ABI compatibility issues.

### 1.2. Install ADFR

#### Linux

Visit the following website and install based on your environment:

https://ccsb.scripps.edu/adfr/downloads/

#### MacOS

If you are using macOS, the macOS safety feature may prevent the installation. You can install with the provided install script `install_ADFR_mac.sh` (Only works for macOS). You might need to give it permission to run first:

```
chmod +x install_ADFR_mac.sh
```

Then run:

```
./install_ADFR_mac.sh
```

> **Note:** ADFRsuite provides a fully fledged Python interpreter (v2.7) that will be installed in the ADFRsuite-1.0 folder. This Python interpreter is insulated from, and will not interfere with, the default python interpreter on your computer. It can be invoked using ADFRsuite-1.0/bin/pythonsh.

> **WARNING:** DO NOT add `ADFRsuite-1.0` or `ADFRsuite-1.0/bin` to your system `PATH` variable. Otherwise, commandline `python` may  call this `python2.7`.




---
## Part 2: Run the ionerdss pipeline 

Same as in the previous tutorial, we set up and run the ionerdss pipeline. Note that we turn on ProAffinity-GNN prediction by giving the `predict_affinity=True` flag and providing the `adfr_path`.

> **WARNING!** The following block takes 10 - 30 minutes to run depending on your hardware!

> **WARNING!** Enabling ProAffinity-GNN can be heavily time- and memory-consuming. 

In [ ]:
#  Path handling (standard library)
from pathlib import Path

# Core imports
import ionerdss as ion
from ionerdss import build_system_from_pdb

# For visualizations
import pandas as pd
import matplotlib.pyplot as plt

pdb_id = "8y7s"

# Build the system using simplified API
# This should take ~5 seconds for 8y7s
system = build_system_from_pdb(
    source=pdb_id,
    workspace_path=f"{pdb_id}_dir",
    interface_detect_distance_cutoff=1.5,
    interface_detect_n_residue_cutoff=5,
    chain_grouping_seq_threshold=0.5,

    # nerdss
    nerdss_total_molecule_count = 40,
    nerdss_n_itr = 100000,
    nerdss_water_box=[188, 188, 188], # = 10 uM

    # ProAffinity
    predict_affinity=True,  # Enable affinity prediction
    adfr_path='~/Documents/ADFR',  # Path to ADFR
    
    # ODE Pipeline Configuration
    ode_enabled=True,            # Now using System-compatible generator!
    ode_time_span=None,   # Auto-calculated based on NERDSS simulation time
    ode_solver_method="BDF",     # Solver for stiff systems
    ode_plot=True,               # Generate plots
    ode_save_csv=True            # Save data to CSV
)

# (Optional) List out generated files
# List all generated files
workspace_path = Path(f"{pdb_id}_dir")

print("Generated Files:")

print("\n NERDSS Input Files:")
nerdss_dir = workspace_path / "nerdss_files"
if nerdss_dir.exists():
    for file in sorted(nerdss_dir.glob("*.mol")) + sorted(nerdss_dir.glob("*.inp")):
        size = file.stat().st_size / 1024  # KB
        print(f"  nerdss_files/{file.name:<30} ({size:>6.1f} KB)")

print("\n System Data:")
outputs_dir = workspace_path / "outputs" / "systems"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.json")):
        size = file.stat().st_size / 1024  # KB
        print(f"  outputs/systems/{file.name:<27} ({size:>6.1f} KB)")

print("\n System Builder Log:")
outputs_dir = workspace_path / "logs"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.log")):
        size = file.stat().st_size / 1024  # KB
        print(f"  logs/{file.name:<38} ({size:>6.1f} KB)") 

---
## Part 2: Run NERDSS Simulation

### Run NERDSS with python subprocess

> **Note**: This requires NERDSS to be installed on your system. The user also has to specify the path to the NERDSS executable.

In [ ]:
# run NERDSS with subprocess
import subprocess

# Check if NERDSS is available
# nerdss_cmd should be replaced with the actual path to the NERDSS executable
nerdss_cmd = "PATH_TO_NERDSS_REPO/bin/nerdss"
nerdss_path = Path(nerdss_cmd).expanduser() # replaces tilde with appropriate user home path

if nerdss_path.exists():
    
    # Run NERDSS
    result = subprocess.run(
        f"{nerdss_cmd} -f parms.inp",
        shell=True,
        cwd=f"{pdb_id}_dir/nerdss_files",
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✓ NERDSS simulation completed!")
        print(f"\nCheck {pdb_id}_dir/nerdss_files/ for output files")
    else:
        print("⚠ NERDSS simulation failed")
        print(result.stderr[:500])
else:
    print("⚠ NERDSS not found at:", nerdss_cmd)

---
## Part 3: Analyze NERDSS Output

After running NERDSS simulations, we can analyze the results using the `Analyzer` class.


In [ ]:
# Initialize Analyzer with NERDSS output directory
analysis = ion.Analyzer(f"{pdb_id}_dir")

# Display discovered simulations
print(f"Found {len(analysis.simulations)} simulation(s)")
for i, sim in enumerate(analysis.simulations):
    print(f"  [{i}] Simulation ID: {sim.id}")

#################
# Plot the NERDSS trajectory alone
plt.figure()

sim = analysis.get_simulation(0)
complex_compositions = [{"A":1},
                        {"A":2},
                        {"A":3},
                        {"A":4}] 

# get the time series data for the above complexes
time, counts = sim.get_time_series(complex_compositions)

print(sim.get_time_series(complex_compositions))

# plot all the returned data
for i in range(len(complex_compositions)):
    plt.plot(time,counts[i],label=str(complex_compositions[i]))

plt.legend()
plt.show()

---
## Summary and Next Steps

### What We've Done

 Loaded 8Y7S structure  
 Predicted the affinity of the binding site with ProAffinity-GNN  
 Exported NERDSS simulation files  
 Ran NERDSS simulation for the assembly

---

## Reference

(1) ProAffinity-GNN: A Novel Approach to Structure-Based Protein–Protein Binding Affinity Prediction via a Curated Data Set and Graph Neural Networks
Zhiyuan Zhou, Yueming Yin, Hao Han, Yiping Jia, Jun Hong Koh, Adams Wai-Kin Kong, and Yuguang Mu Journal of Chemical Information and Modeling 2024 64 (23), 8796-8808
DOI: 10.1021/acs.jcim.4c01850

(2) Pradeep Anand Ravindranath, Stefano Forli, David S. Goodsell, Arthur J. Olson, Michel F. Sanner (2015) AutoDockFR: Advances in Protein-Ligand Docking with Explicitly Specified Binding Site Flexibility. PLOS Computational Biology 11(12): e1004586. https://doi.org/10.1371/journal.pcbi.1004586

(3) Yong Zhao Daniel Stoffler Michel Sanner (2006) Hierarchical and multi-resolution representation of protein flexibility. Bioinformatics, Volume 22, Issue 22, 15 November 2006, Pages 2768–2774, https://doi.org/10.1093/bioinformatics/btl481

---

*Tutorial created: 2026-2-4*  
*IONERDSS Version: 1.2.0*